# **CLIP Leave-One-Out Generalization Experiment**
For each dataset: train on that dataset only, evaluate on all 5 datasets.
Results saved back to inventory as two columns per trained model:
  {short_name}_pred_label, {short_name}_pred_confident

In [1]:
# Block 0 - Mount and install
from google.colab import drive
drive.mount('/content/drive')
!pip install -q transformers

Mounted at /content/drive


In [2]:
# Block 1 - Imports, config, shared utilities
import os
import shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from transformers import CLIPVisionModel

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

INVENTORY_PATH = "/content/drive/MyDrive/TrainingData/dataset_inventory.csv"
MODEL_PATH     = "/content/drive/MyDrive/Model/clip_model"
SAVE_DIR       = "/content/drive/MyDrive/Model/clip_leave_one_out"
LOCAL_IMG_DIR  = "/content/local_images"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOCAL_IMG_DIR, exist_ok=True)

BATCH_SIZE = 128
EPOCHS     = 5
LR         = 2e-5
VAL_SPLIT  = 0.15
SEED       = 42

# Short names used for column naming in inventory
DATASETS = {
    'imagenet_ai_0424_sdv5':   'sdv5',
    'imagenet_ai_0419_vqdm':   'vqdm',
    'imagenet_ai_0508_adm':    'adm',
    'imagenet_ai_0419_biggan': 'biggan',
    'imagenet_glide':          'glide',
}

torch.manual_seed(SEED)
print(f"Device: {DEVICE}")

clip_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275, 0.40821073],
        std=[0.26862954, 0.26130258, 0.27577711]
    )
])

Device: cuda


In [9]:
# Block 2 - Copy trainval images to VM
# Only copies images tagged as 'trainval' split for the 5 target datasets
df_full = pd.read_csv(INVENTORY_PATH)

df_all = df_full[
    (df_ful
     l['dataset'].isin(DATASETS.keys())) &
    (df_full['split'] == 'trainval')
].copy().reset_index(drop=True)

print(f"Trainval samples to copy: {len(df_all):,}")
print(df_all.groupby(['dataset', 'is_fake']).size().to_string())

def copy_single(args):
    idx, row = args
    dest = os.path.join(LOCAL_IMG_DIR, f"{idx}_{row['file_name']}")
    if not os.path.exists(dest):
        try:
            shutil.copy2(row['file_path'], dest)
        except Exception:
            return None
    return dest

print("Copying trainval images to VM...")
with ThreadPoolExecutor(max_workers=16) as executor:
    local_paths = list(tqdm(
        executor.map(copy_single, df_all.iterrows()),
        total=len(df_all)
    ))

df_all['local_path'] = local_paths
df_all = df_all.dropna(subset=['local_path']).reset_index(drop=True)
print(f"Ready: {len(df_all):,} trainval images on VM")

In [3]:
# Block 3 - Model architecture and helper functions
class ClassificationCLIP(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.vision_encoder = CLIPVisionModel.from_pretrained(model_path)
        hidden_size = self.vision_encoder.config.hidden_size
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, pixel_values):
        outputs = self.vision_encoder(pixel_values=pixel_values)
        return self.classifier(outputs.pooler_output)


class ImageDataset(Dataset):
    def __init__(self, df, transform, path_col='local_path'):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.path_col  = path_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = float(1 - row['is_fake'])  # is_fake=1->0(fake), is_fake=0->1(real)
        try:
            img = Image.open(row[self.path_col]).convert('RGB')
            img = self.transform(img)
        except Exception:
            img = torch.zeros(3, 224, 224)
        return img, torch.tensor([label], dtype=torch.float32), row['dataset']


def build_model():
    """Build a fresh model with last 2 layers unfrozen."""
    m = ClassificationCLIP(MODEL_PATH).to(DEVICE)
    for param in m.vision_encoder.parameters():
        param.requires_grad = False
    for layer in m.vision_encoder.vision_model.encoder.layers[-2:]:
        for param in layer.parameters():
            param.requires_grad = True
    for param in m.classifier.parameters():
        param.requires_grad = True
    return m


@torch.no_grad()
def validate(model, loader):
    model.eval()
    correct = total = 0
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(imgs)
        preds    = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return correct / total


@torch.no_grad()
def run_eval(model, df_eval, desc='Evaluating'):
    """Run inference on df_eval.
    Returns per-dataset accuracy report and a prob list aligned to df_eval index.
    """
    model.eval()

    class IndexedDataset(Dataset):
        def __init__(self, df, transform):
            self.df        = df.reset_index(drop=True)
            self.transform = transform
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            row   = self.df.iloc[idx]
            label = float(1 - row['is_fake'])
            try:
                img = Image.open(row['local_path']).convert('RGB')
                img = self.transform(img)
            except Exception:
                img = torch.zeros(3, 224, 224)
            return img, torch.tensor([label], dtype=torch.float32), row['dataset'], idx

    ds = IndexedDataset(df_eval, clip_transform)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=4, pin_memory=True)

    # Pre-allocate prob array aligned to df_eval
    all_probs = np.full(len(df_eval), np.nan)
    results   = defaultdict(lambda: {'correct': 0, 'total': 0,
                                      'real_correct': 0, 'real_total': 0,
                                      'fake_correct': 0, 'fake_total': 0})

    for imgs, labels, datasets, indices in tqdm(dl, desc=desc):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(imgs)
        prob_fake = (1 - torch.sigmoid(logits)).squeeze(1).cpu().float().numpy()
        preds     = (torch.sigmoid(logits) >= 0.5).float()
        correct   = (preds == labels)

        # Store probs at correct positions
        all_probs[indices.numpy()] = prob_fake

        for i in range(len(labels)):
            d     = datasets[i]
            is_fk = int(1 - labels[i].item())
            ok    = correct[i].item()
            results[d]['total']   += 1
            results[d]['correct'] += ok
            if is_fk:
                results[d]['fake_total']   += 1
                results[d]['fake_correct'] += ok
            else:
                results[d]['real_total']   += 1
                results[d]['real_correct'] += ok

    assert not np.isnan(all_probs).any(), \
        f"Missing predictions: {np.isnan(all_probs).sum()} rows not covered"

    return results, all_probs.tolist()


def print_report(train_dataset_name, results):
    print(f"\n{'='*65}")
    print(f"Trained on: {train_dataset_name}")
    print(f"{'Dataset':<30} {'Overall':>8} {'Real':>8} {'Fake':>8}")
    print('-' * 65)
    total_c = total_a = 0
    for ds_name in DATASETS.keys():
        r       = results[ds_name]
        overall = r['correct'] / r['total'] * 100 if r['total'] else 0
        real    = r['real_correct'] / r['real_total'] * 100 if r['real_total'] else 0
        fake    = r['fake_correct'] / r['fake_total'] * 100 if r['fake_total'] else 0
        marker  = ' <-- trained on this' if ds_name == train_dataset_name else ''
        print(f"{ds_name:<30} {overall:>7.2f}%  {real:>7.2f}%  {fake:>7.2f}%{marker}")
        total_c += r['correct']
        total_a += r['total']
    print('-' * 65)
    print(f"{'TOTAL':<30} {total_c/total_a*100:>7.2f}%")
    print('=' * 65)


def save_predictions_to_inventory(df_eval, probs, short_name):
    """Write pred_label and pred_confident columns back to inventory CSV."""
    label_col = f"{short_name}_pred_label"
    conf_col  = f"{short_name}_pred_confident"

    df_eval = df_eval.copy()
    df_eval[label_col] = [1 if p >= 0.5 else 0 for p in probs]
    df_eval[conf_col]  = [round(p, 4) for p in probs]

    df_inv = pd.read_csv(INVENTORY_PATH)
    if label_col not in df_inv.columns:
        df_inv[label_col] = None
    if conf_col not in df_inv.columns:
        df_inv[conf_col] = None

    df_inv = df_inv.set_index('file_path')
    df_e   = df_eval.set_index('file_path')
    df_inv.loc[df_e.index, label_col] = df_e[label_col]
    df_inv.loc[df_e.index, conf_col]  = df_e[conf_col]
    df_inv = df_inv.reset_index()
    df_inv.to_csv(INVENTORY_PATH, index=False)
    print(f"Saved columns '{label_col}', '{conf_col}' -> {INVENTORY_PATH}")


print("All utilities ready.")

All utilities ready.


In [ ]:
# Block 4 - Leave-one-out training loop
# For each dataset:
#   1. Train on that dataset (trainval split)
#   2. Eval on ALL 5 datasets (test split)
#   3. Save predictions to inventory

criterion = nn.BCEWithLogitsLoss()

# Test split comes from inventory directly (Drive paths)
# local_path is added by copying test images to VM
df_inv      = pd.read_csv(INVENTORY_PATH)
df_test_all = df_inv[
    (df_inv['dataset'].isin(DATASETS.keys())) &
    (df_inv['split'] == 'test')
].copy().reset_index(drop=True)

print(f"Test pool: {len(df_test_all):,} images")
print(df_test_all.groupby(['dataset', 'is_fake']).size().to_string())

# Copy test images to VM once (reused across all 5 training runs)
LOCAL_TEST_DIR = '/content/local_test_images'
os.makedirs(LOCAL_TEST_DIR, exist_ok=True)

def copy_test(args):
    idx, row = args
    dest = os.path.join(LOCAL_TEST_DIR, f"{idx}_{row['file_name']}")
    if not os.path.exists(dest):
        try:
            shutil.copy2(row['file_path'], dest)
        except Exception:
            return None
    return dest

print("Copying test images to VM...")
with ThreadPoolExecutor(max_workers=16) as executor:
    test_local_paths = list(tqdm(
        executor.map(copy_test, df_test_all.iterrows()),
        total=len(df_test_all)
    ))

df_test_all['local_path'] = test_local_paths
df_test_all = df_test_all.dropna(subset=['local_path']).reset_index(drop=True)
print(f"Test images on VM: {len(df_test_all):,}")

# Verify df_all (trainval) also has local_path from Block 2
assert 'local_path' in df_all.columns, "Run Block 2 first"
assert df_all['local_path'].notna().all(), "Some trainval images missing local_path, re-run Block 2"

for train_ds_name, short_name in DATASETS.items():
    print(f"\n{'#'*65}")
    print(f"TRAINING ON: {train_ds_name}")
    print(f"{'#'*65}")

    # Build fresh model for each run
    model     = build_model()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=1e-2
    )
    scaler    = torch.amp.GradScaler('cuda')
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2
    )

    # Build train/val loaders from this dataset only (local_path on VM)
    df_trainval = df_all[
        (df_all['dataset'] == train_ds_name) &
        (df_all['split'] == 'trainval')
    ].copy().reset_index(drop=True)

    full_ds = ImageDataset(df_trainval, clip_transform)
    val_n   = int(len(full_ds) * VAL_SPLIT)
    train_n = len(full_ds) - val_n
    train_ds_split, val_ds_split = random_split(
        full_ds, [train_n, val_n],
        generator=torch.Generator().manual_seed(SEED)
    )
    train_loader = DataLoader(train_ds_split, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, persistent_workers=True)
    val_loader   = DataLoader(val_ds_split,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True, persistent_workers=True)

    print(f"Train: {train_n:,} | Val: {val_n:,}")

    # Training
    best_val_acc = 0.0
    ckpt_path    = os.path.join(SAVE_DIR, f"best_{short_name}.pth")

    for epoch in range(EPOCHS):
        model.train()
        run_loss = correct = total = 0

        bar = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{EPOCHS}")
        for imgs, labels, _ in bar:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(imgs)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item()
            preds     = (torch.sigmoid(logits.detach()) >= 0.5).float()
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)
            bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct/total:.2%}")

        val_acc = validate(model, val_loader)
        scheduler.step(val_acc)
        print(f"  Epoch {epoch+1}  loss={run_loss/len(train_loader):.4f}  "
              f"train={correct/total:.2%}  val={val_acc:.2%}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), ckpt_path)
            print(f"  Best saved: val={best_val_acc:.2%}")

    # Load best checkpoint
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False))

    # Eval on ALL test datasets (uses local_path on VM)
    results, probs = run_eval(model, df_test_all, desc=f"  Eval all ({short_name})")
    print_report(train_ds_name, results)

    # Save predictions to inventory using original Drive file_path for merge
    save_predictions_to_inventory(df_test_all, probs, short_name)

print(f"\nAll done. Inventory updated at: {INVENTORY_PATH}")

Test pool: 2,000 images
dataset                  is_fake
imagenet_ai_0419_biggan  0          200
                         1          200
imagenet_ai_0419_vqdm    0          200
                         1          200
imagenet_ai_0424_sdv5    0          200
                         1          200
imagenet_ai_0508_adm     0          200
                         1          200
imagenet_glide           0          200
                         1          200
Copying test images to VM...


  0%|          | 0/2000 [00:00<?, ?it/s]

Test images on VM: 2,000

#################################################################
TRAINING ON: imagenet_ai_0424_sdv5
#################################################################


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: /content/drive/MyDrive/Model/clip_model
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
visual_projection.weight                                     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v

...

  Epoch 4  loss=0.0101  train=99.94%  val=98.67%


  Epoch 5/5:   0%|          | 0/14 [00:00<?, ?it/s]

  Epoch 5  loss=0.0027  train=100.00%  val=98.67%


  Eval all (glide):   0%|          | 0/16 [00:00<?, ?it/s]


Trained on: imagenet_glide
Dataset                         Overall     Real     Fake
-----------------------------------------------------------------
imagenet_ai_0424_sdv5            59.25%    98.00%    20.50%
imagenet_ai_0419_vqdm            81.25%    99.50%    63.00%
imagenet_ai_0508_adm             71.25%   100.00%    42.50%
imagenet_ai_0419_biggan          98.75%    99.00%    98.50%
imagenet_glide                   99.00%    99.00%    99.00% <-- trained on this
-----------------------------------------------------------------
TOTAL                            81.90%
Saved columns 'glide_pred_label', 'glide_pred_confident' -> /content/drive/MyDrive/TrainingData/dataset_inventory.csv

All done. Inventory updated at: /content/drive/MyDrive/TrainingData/dataset_inventory.csv


## **EVALUATION**

In [4]:
# --- 1. CONFIGURATION ---
drive.mount('/content/drive')

# Update these paths to your actual locations
MODEL_WEIGHTS  = "/content/drive/MyDrive/Model/clip_classification_v5/best_model.pth"
MODEL_BASE     = "/content/drive/MyDrive/Model/clip_model" # HuggingFace base model path

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE  = 64

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# --- 2. MODEL ARCHITECTURE ---
class ClassificationCLIP(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.vision_encoder = CLIPVisionModel.from_pretrained(model_path)
        hidden_size = self.vision_encoder.config.hidden_size
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, pixel_values):
        outputs = self.vision_encoder(pixel_values=pixel_values)
        return self.classifier(outputs.pooler_output)

In [6]:
# 3. Model Architecture
class ClassificationCLIP(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.vision_encoder = CLIPVisionModel.from_pretrained(model_path)
        hidden_size = self.vision_encoder.config.hidden_size
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, pixel_values):
        outputs = self.vision_encoder(pixel_values=pixel_values)
        return self.classifier(outputs.pooler_output)

In [7]:
# 4. Image Preprocessing
clip_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275, 0.40821073],
        std=[0.26862954, 0.26130258, 0.27577711]
    )
])

# 5. Initialize and Load Weights
print("Initializing Model...")
model = ClassificationCLIP(MODEL_BASE).to(DEVICE)
model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=DEVICE))
model.eval()
print(f"Model loaded successfully on {DEVICE}")

Initializing Model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: /content/drive/MyDrive/Model/clip_model
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v

Model loaded successfully on cuda


In [10]:
# import os
# import pandas as pd
# import torch
# from torch.utils.data import Dataset, DataLoader
# from PIL import Image
# from tqdm import tqdm
# from collections import defaultdict

# # --- 1. SETTINGS ---
# MODEL_FILES = ['best_adm.pth', 'best_biggan.pth', 'best_glide.pth', 'best_model.pth', 'best_sdv5.pth', 'best_vqdm.pth']
# MODELS_DIR = '/content/drive/MyDrive/Model/clip_classification_v5/'
# INVENTORY_PATH = "/content/drive/MyDrive/TrainingData/dataset_inventory_fix.csv"
# BATCH_SIZE = 128
# NUM_WORKERS = os.cpu_count()

# # --- 2. DATASET CLASS ---
# class TestDataset(Dataset):
#     def __init__(self, df, transform):
#         self.df = df.reset_index(drop=True)
#         self.transform = transform
#     def __len__(self): return len(self.df)
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         label = float(1 - row['is_fake']) # 1 = Real, 0 = Fake
#         try:
#             img = Image.open(row['file_path']).convert('RGB')
#             img = self.transform(img)
#         except Exception:
#             img = torch.zeros(3, 224, 224)
#         return img, torch.tensor([label], dtype=torch.float32), row['dataset'], int(row['is_fake']), row['file_path']

# # Load Inventory
# df_full = pd.read_csv(INVENTORY_PATH)

# # --- 3. EVALUATION LOOP ---
# for model_file in MODEL_FILES:
#     print("\n" + "="*80)
#     print(f" CURRENTLY EVALUATING MODEL: {model_file.upper()}")
#     print("="*80)

#     # Load Model Weights (Assumes 'model' and 'DEVICE' are already defined)
#     model_path = os.path.join(MODELS_DIR, model_file)
#     try:
#         model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=False))
#         model.eval()
#     except Exception as e:
#         print(f" Error loading {model_file}: {e}")
#         continue

#     # Prepare Test Data
#     df_test = df_full[(df_full['dataset'].isin(DATASETS)) & (df_full['split'] == 'test')].copy().reset_index(drop=True)
#     test_ds = TestDataset(df_test, clip_transform)
#     test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

#     # Storage for stats and results
#     results_summary = defaultdict(lambda: {'correct': 0, 'total': 0, 'real_correct': 0, 'real_total': 0, 'fake_correct': 0, 'fake_total': 0})
#     all_preds, all_confidents, all_paths = [], [], []

#     # Inference Mode
#     with torch.inference_mode():
#         for imgs, labels, datasets, is_fakes, paths in tqdm(test_dl, desc=f"Inference [{model_file}]"):
#             imgs = imgs.to(DEVICE, non_blocking=True)
#             with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
#                 logits = model(imgs)

#             prob_real = torch.sigmoid(logits).squeeze(1).cpu().float().numpy()

#             for i in range(len(prob_real)):
#                 p_real = prob_real[i]
#                 pred_label = 1 if p_real >= 0.5 else 0 # 1=Real, 0=Fake
#                 conf = p_real if pred_label == 1 else (1 - p_real)

#                 all_preds.append(pred_label)
#                 all_confidents.append(round(float(conf), 4))
#                 all_paths.append(paths[i])

#                 # Metrics Calculation
#                 ds = datasets[i]
#                 true_is_fake = is_fakes[i]
#                 is_correct = (pred_label == (1 - true_is_fake))

#                 results_summary[ds]['total'] += 1
#                 results_summary[ds]['correct'] += int(is_correct)
#                 if true_is_fake:
#                     results_summary[ds]['fake_total'] += 1
#                     results_summary[ds]['fake_correct'] += int(is_correct)
#                 else:
#                     results_summary[ds]['real_total'] += 1
#                     results_summary[ds]['real_correct'] += int(is_correct)

#     # --- 4. PRINT SUMMARY REPORT ---
#     print("\n" + "-" * 65)
#     print(f" REPORT FOR: {model_file}")
#     print(f"{'Dataset':<30} {'Overall':>8} {'Real':>8} {'Fake':>8}")
#     print("-" * 65)

#     total_correct = total_all = 0
#     for ds in DATASETS:
#         r = results_summary[ds]
#         if r['total'] == 0: continue
#         overall = r['correct'] / r['total'] * 100
#         real    = r['real_correct'] / r['real_total'] * 100 if r['real_total'] else 0
#         fake    = r['fake_correct'] / r['fake_total'] * 100 if r['fake_total'] else 0
#         total_correct += r['correct']
#         total_all     += r['total']
#         print(f"{ds:<30} {overall:>7.2f}%  {real:>7.2f}%  {fake:>7.2f}%")

#     print("-" * 65)
#     if total_all > 0:
#         print(f"{'TOTAL AVG':<30} {total_correct/total_all*100:>7.2f}%")
#     print("-" * 65)

#     # --- 5. SAVE TO INVENTORY ---
#     model_suffix = model_file.replace('best_', '').replace('.pth', '')
#     label_col = f"{model_suffix}_pred_label"
#     conf_col = f"{model_suffix}_pred_confident"

#     df_results = pd.DataFrame({'file_path': all_paths, label_col: all_preds, conf_col: all_confidents}).set_index('file_path')
#     df_full = df_full.set_index('file_path')

#     if label_col not in df_full.columns:
#         df_full[label_col] = None
#         df_full[conf_col] = None

#     df_full.update(df_results)
#     df_full = df_full.reset_index()
#     df_full.to_csv(INVENTORY_PATH, index=False, encoding='utf-8-sig')
#     print(f"Results saved to {INVENTORY_PATH}\n")

# print(f"\n ALL EVALUATIONS FINISHED!")